# E7 step 1: PEM with the road prior, then the E7 graph (Kaggle, CPU)

Runs `run_local_pem.py --prior road` on the VAR training residuals (same settings as E4; only the
prior changes: 0.7 on road links, 0.3 elsewhere instead of 0.5), then `build_graph.py E7`, and
compares the E7 graph with E4. The E7 A3T-GCN training is step 2, in `ablation-study.ipynb`.

**Before running**
1. *Add Input* -> the GBA dataset (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add Input* -> the artifacts dataset (needs `E4__...npz` for the comparison).
3. *Add Input* -> the **PEM inputs dataset**: `residuals.npy`, `metadata.json`, `sensors.npy`,
   `coefficients.npy` (from `outputs/var_train/lag3_alpha1e+06/`) and `lingam_result.npz`
   (from `outputs/lingam/lag3_alpha1e+06__rows10000/`).
4. Settings: **Accelerator = None (CPU)**, **Internet = On**.
5. **Save Version -> Save & Run All (Commit)**.

**If the run stops before all 2,056 units are done** (12-hour limit): the finished units are in this
version's output. Open a new version, *Add Input* -> *Notebook Output* of this notebook, and run again:
finished units are copied back and skipped.

Download from the *Output* tab: `e7_output/` (E7 graph `.npz` + `.json`, `pem_result.npz`, `summary.json`,
`e7_vs_e4.json`).

## Settings

In [ ]:
import os

REPO_URL    = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH = "main"

WORKERS            = os.cpu_count()   # Kaggle CPU session: 4
TIME_BUDGET_HOURS  = 11.0             # stop PEM before Kaggle's 12-hour limit; finished units are kept

## 1. Environment

In [ ]:
import sys, numpy as np, scipy
print("Python:", sys.version)
print("NumPy:", np.__version__, "| SciPy:", scipy.__version__)
print("CPUs:", os.cpu_count())

## 2. Code, dataset and PEM inputs

In [ ]:
import glob, shutil, subprocess

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# GBA dataset -> implementation/datasets/LargeST GBA
hits = glob.glob("/kaggle/input/**/gba_his_raw_2021.h5", recursive=True)
assert hits, "Add the GBA dataset (Add Input)"
os.makedirs(f"{CODE}/datasets", exist_ok=True)
os.symlink(os.path.dirname(hits[0]), f"{CODE}/datasets/LargeST GBA")


def find_one(name):
    found = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)
             if "/units/" not in p and "/implementation/" not in p]
    assert len(found) == 1, f"expected exactly one {name} under /kaggle/input, found {found}"
    return found[0]


# Folder names must match config.py (Kaggle drops "+" from uploaded names, so they are created here).
VAR_DIR    = f"{CODE}/outputs/var_train/lag3_alpha1e+06"
LINGAM_DIR = f"{CODE}/outputs/lingam/lag3_alpha1e+06__rows10000"
GRAPH_DIR  = f"{CODE}/outputs/graphs"
for d in (VAR_DIR, LINGAM_DIR, GRAPH_DIR):
    os.makedirs(d, exist_ok=True)

for name in ("residuals.npy", "metadata.json", "sensors.npy", "coefficients.npy"):
    os.symlink(find_one(name), f"{VAR_DIR}/{name}")        # symlinks: large files are not copied
os.symlink(find_one("lingam_result.npz"), f"{LINGAM_DIR}/lingam_result.npz")

for path in glob.glob("/kaggle/input/**/E4__*", recursive=True):
    shutil.copy(path, f"{GRAPH_DIR}/{os.path.basename(path).replace('alpha1e06', 'alpha1e+06')}")

for p in sorted(glob.glob(f"{CODE}/outputs/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p) / 1e6:9.1f} MB  {p.replace(CODE + '/', '')}")

## 3. Settings check (must equal E4's PEM run except the prior)

In [ ]:
sys.path.insert(0, CODE)
from config import config as C

print("VAR run    :", C.VAR_RUN)
print("LiNGAM run :", C.LINGAM_RUN)
print("E4 PEM run :", C.PEM_RUN)
print("E7 PEM run :", C.PEM_RUN_E7)
print("E7 graph   :", C.FINAL_GRAPHS["E7"])

assert (C.PEM_NU0, C.PEM_NU1, C.PEM_TAU, C.PEM_THRESHOLD) == (0.0017, 1.0, 0.01, 0.5)
assert C.PEM_STANDARDISE and C.PEM_OPTIMIZER == "L-BFGS-B"
assert (C.PEM_PRIOR_HIGH, C.PEM_PRIOR_LOW) == (0.7, 0.3)
assert C.PEM_RUN_E7 == C.PEM_RUN.replace("eta_0.5", "eta_road0.7-0.3")

PEM_DIR = f"{CODE}/outputs/pem/{C.PEM_RUN_E7}"
UNIT_DIR = f"{PEM_DIR}/units"

## 4. Resume: copy finished units from an earlier version's output (if added as input)

In [ ]:
os.makedirs(UNIT_DIR, exist_ok=True)
previous = [p for p in glob.glob("/kaggle/input/**/unit_*.npz", recursive=True)
            if "e7_units__" in p and "eta_road0.7-0.3" in p and "nu0_0.0017" in p]
for p in previous:
    shutil.copy(p, UNIT_DIR)
print("units copied from an earlier run:", len(previous))

## 5. Run PEM (road prior) with a time budget

In [ ]:
import time

cmd = [sys.executable, "-u", "run_local_pem.py", "--prior", "road", "--workers", str(WORKERS)]
print(" ".join(cmd))

start = time.time()
log = open(f"{WORK}/pem_e7.log", "w")
proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
last = 0
for line in proc.stdout:
    log.write(line); log.flush()
    if time.time() - last > 300 or "error" in line.lower() or "saved" in line.lower():
        print(f"[{(time.time() - start) / 3600:5.2f} h] {line.rstrip()}", flush=True)
        last = time.time()
    if time.time() - start > TIME_BUDGET_HOURS * 3600:
        proc.kill()
        print("Time budget reached: PEM stopped; finished units are kept.")
        break
proc.wait()
log.close()

done = len(glob.glob(f"{UNIT_DIR}/unit_*.npz"))
PEM_COMPLETE = os.path.exists(f"{PEM_DIR}/pem_result.npz")
print(f"units finished: {done} / 2056 | pem_result.npz written: {PEM_COMPLETE} | "
      f"hours: {(time.time() - start) / 3600:.2f}")
print("".join(open(f"{WORK}/pem_e7.log").readlines()[-25:]))

## 6. Build the E7 graph (only when PEM finished)

In [ ]:
if PEM_COMPLETE:
    r = subprocess.run([sys.executable, "build_graph.py", "E7"], cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-4000:], r.stderr[-4000:])
    assert r.returncode == 0
else:
    print("PEM not finished: run a new version with this version's output added as input (section 4).")

## 7. E7 vs E4: how much does the road prior change the graph?

In [ ]:
import json

if PEM_COMPLETE:
    with np.load(f"{GRAPH_DIR}/{C.FINAL_GRAPHS['E7']}") as a:
        e7, sensors = a["adjacency"].astype(np.float64), a["sensors"]
    with np.load(f"{GRAPH_DIR}/{C.FINAL_GRAPHS['E4']}") as a:
        e4 = a["adjacency"].astype(np.float64)
        assert np.array_equal(a["sensors"], sensors)

    road = np.load(f"{CODE}/datasets/LargeST GBA/gba_rn_adj.npy", mmap_mode="r")
    road = np.asarray(road[np.ix_(sensors, sensors)]) > 0
    road = road | road.T
    off = ~np.eye(len(sensors), dtype=bool)

    def share(g):   # median own-signal share after A3T-GCN normalisation
        a = g.T + np.eye(len(g)); d = a.sum(1)
        return float(np.median(1.0 / d))

    diff = np.abs(e7 - e4)
    res = {
        "edges_E4": int(((e4 != 0) & off).sum()),
        "edges_E7": int(((e7 != 0) & off).sum()),
        "edges_only_E4": int(((e4 != 0) & (e7 == 0) & off).sum()),
        "edges_only_E7": int(((e7 != 0) & (e4 == 0) & off).sum()),
        "entries_changed": int(((diff > 1e-9) & off).sum()),
        "total_weight_change_pct": float(100 * diff.sum() / e4.sum()),
        "entries_changed_over_10pct": int(((diff > 0.1 * np.maximum(e4, 1e-12)) & off & (e4 > 0)).sum()),
        "weight_change_on_road_pairs": float((e7 - e4)[road & off].sum()),
        "weight_change_on_non_road_pairs": float((e7 - e4)[~road & off].sum()),
        "own_signal_share_E4": share(e4),
        "own_signal_share_E7": share(e7),
    }
    print(json.dumps(res, indent=2))
    json.dump(res, open(f"{WORK}/e7_vs_e4.json", "w"), indent=2)

## 8. Collect the outputs to download

In [ ]:
OUT = f"{WORK}/e7_output"
os.makedirs(OUT, exist_ok=True)
for p in [f"{PEM_DIR}/pem_result.npz", f"{PEM_DIR}/summary.json", f"{WORK}/e7_vs_e4.json", f"{WORK}/pem_e7.log",
          *glob.glob(f"{GRAPH_DIR}/E7__*")]:
    if os.path.exists(p):
        shutil.copy(p, OUT)
print(sorted(os.listdir(OUT)))
print("Upload the E7__...npz (and .json) to the artifacts dataset, then run E7 in ablation-study.ipynb.")

## 9. Clean up

Deletes the cloned `implementation` folder so the Output tab holds only `e7_output/`. If PEM did not
finish, the finished units are kept in `e7_units__<run>/` for the next version (section 4).

In [ ]:
if not PEM_COMPLETE:
    shutil.copytree(UNIT_DIR, f"{WORK}/e7_units__{C.PEM_RUN_E7}", dirs_exist_ok=True)
shutil.rmtree(CODE)
print("Output contents:", sorted(os.listdir(WORK)))